# Module 5 — Inverse Kinematics
## Unit 4 — From Geometry to Numerical IK
### Lesson 4.4 — From Geometry to Numerical IK (Unit 4 Recap · Midpoint)

*Physical AI Curriculum · hands-on notebook. Run **Kernel → Restart & Run All**.*


## Midpoint synthesis
Analytical and numerical solvers agree on the same target.


In [ ]:
import numpy as np
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt

def fk_two_link(theta1, theta2, L1, L2):
    x = L1*np.cos(theta1) + L2*np.cos(theta1+theta2)
    y = L1*np.sin(theta1) + L2*np.sin(theta1+theta2)
    return np.array([x, y])

def ik_2link_closed(x, y, L1, L2):
    """Closed-form: return both (theta1,theta2); [] unreachable; one on boundary."""
    c2 = (x*x + y*y - L1*L1 - L2*L2)/(2*L1*L2)
    if c2 < -1-1e-9 or c2 > 1+1e-9:
        return []
    c2 = max(-1.0, min(1.0, c2))
    sols=[]
    for sign in (+1.0, -1.0):
        s2 = sign*np.sqrt(max(0.0, 1.0 - c2*c2))
        t2 = np.arctan2(s2, c2)
        t1 = np.arctan2(y, x) - np.arctan2(L2*np.sin(t2), L1 + L2*np.cos(t2))
        sols.append((t1, t2))
        if abs(s2) < 1e-6:
            break
    return sols

def jacobian_2link(theta1, theta2, L1, L2):
    s1, s12 = np.sin(theta1), np.sin(theta1+theta2)
    c1, c12 = np.cos(theta1), np.cos(theta1+theta2)
    return np.array([[-L1*s1 - L2*s12, -L2*s12],
                     [ L1*c1 + L2*c12,  L2*c12]])

def ik_numerical(target, theta0, L1, L2, alpha=0.5, tol=1e-6, max_iter=200):
    theta=np.array(theta0,float); target=np.array(target,float)
    for it in range(max_iter):
        e=target-fk_two_link(theta[0],theta[1],L1,L2)
        if np.linalg.norm(e)<tol: return theta
        J=jacobian_2link(theta[0],theta[1],L1,L2)
        theta=theta+alpha*np.linalg.solve(J,e)
    return theta

L1,L2=0.4,0.3; target=(0.5,0.2)
# analytical: both solutions
ana=ik_2link_closed(*target,L1,L2)
# numerical: seed near the elbow-down analytical solution
num=ik_numerical(target, np.radians([10,20]), L1,L2)
print("analytical solutions:",[tuple(np.round(np.degrees(s),2)) for s in ana])
print("numerical solution  :",tuple(np.round(np.degrees(num),2)))


## Check your work


In [ ]:
# numerical solution matches one of the analytical ones (FK and angle)
assert np.allclose(fk_two_link(num[0],num[1],L1,L2),target,atol=1e-5)
def near(a,b): return min(np.linalg.norm(np.array(a)-np.array(s)) for s in b)
assert near(num, ana) < 1e-3
print("All checks passed.")
